# IoT-ASP Colab ETL stub

Public notebook — **no site PII**. Secrets via Colab `userdata` only.
Pipeline: GCS telemetry → features → Gemini Enterprise (`iot-asp-autoroute`) / ADK agent.
See `docs/colab-gemini-pipeline.md`.


In [ ]:
from google.colab import userdata
import json, os
from google.oauth2 import service_account
from google.cloud import storage

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "bear-iot-asp-rec")
ENGINE_ID = os.environ.get("IOT_ASP_GEMINI_ENGINE_ID", "iot-asp-autoroute")
BUCKET = userdata.get("IOT_ASP_GCS_BUCKET")
sa_json = userdata.get("GCP_SA_JSON")
creds = service_account.Credentials.from_service_account_info(json.loads(sa_json))
client = storage.Client(project=PROJECT, credentials=creds)
print("ok", PROJECT, ENGINE_ID, BUCKET)


In [ ]:
def latest_telemetry(node_id: str):
    blobs = sorted(
        client.list_blobs(BUCKET, prefix=f"meta/telemetry/{node_id}/"),
        key=lambda b: b.name,
        reverse=True,
    )
    return json.loads(blobs[0].download_as_text()) if blobs else None

def vib_features(t):
    if not t:
        return {}
    abs_a = float(t.get("absA") or 0)
    mic = float(t.get("micEnergy") or 0)
    return {
        "peakHz": t.get("peakHz"),
        "absA": abs_a,
        "micEnergy": mic,
        "vibClass": t.get("vibClass"),
        "priorHint": "structure_borne" if abs_a > mic else "air_borne",
        "engineId": ENGINE_ID,
    }

print("feature stub ready")
